# DRlm


In [1]:
from DRlm import DRlm
import numpy as np
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings("ignore")

## Regression

## High-dimensional Case

### Data Generating Process

In [2]:
np.random.seed(0)  # For reproducibility
# number of groups
L = 2
# dimension
p = 100

# mean vector for source
mean_source = np.zeros(p)

# covariance matrix for source
def A1gen(rho, p):
    A1 = np.zeros((p, p))
    for i in range(p):
        for j in range(p):
            A1[i, j] = rho ** abs(i - j)
    return A1

cov_source = A1gen(0.6, p)

# 1st group's source data
n1 = 100
X1 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n1)
b1 = np.zeros(p)
b1[0:5] = np.arange(1, 6) / 20
b1[97:100] = [0.5, -0.5, -0.5]
Y1 = X1 @ b1 + np.random.normal(size=n1)

# 2nd group's source data
n2 = 100
X2 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n2)
b2 = np.zeros(p)
b2[5:10] = np.arange(1, 6) / 20
b2[97:100] = 0.5 * np.array([0.5, -0.5, -0.5])
Y2 = X2 @ b2 + np.random.normal(size=n2)

# Target Data, covariate shift
n0 = 100
mean0 = np.zeros(p)
cov0 = cov_source.copy()

# diagonal elements
for i in range(p):
    cov0[i, i] = 1.5

# first 5x5 block off-diagonal
for i in range(5):
    for j in range(5):
        if i != j:
            cov0[i, j] = 0.9

# last 2x2 block off-diagonal (indices 98~100 in R = 97~99 in Python)
for i in range(98, 100):
    for j in range(98, 100):
        if i != j:
            cov0[i, j] = 0.9

X0 = multivariate_normal.rvs(mean=mean0, cov=cov0, size=n0)

Xlist = [X1, X2]
ylist = [Y1, Y2]


In [3]:
# dimension p=100
loading_mat = np.zeros((100, 2))
loading_mat[95:100, 0] = 0.4  
loading_mat[98:100, 1] = 0.8

loading_mat = loading_mat.T


### Estimation ($\widehat{\omega \beta^\mathbb{Q}}, \widehat{\gamma}$)

In [4]:
reg = DRlm.Regression(f_learner='high_d')
reg.fit(Xlist, ylist, loading_mat, X0=X0)
reg.infer(M=500, alpha=0.05, alpha_thres=0.01)

In [5]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.1485   0.8515

Fitted Plug-in Estimations (Maximin Effects):

dimension |        1        2        3        4        5        6        7        8        9       10
coef_     |   0.0074  -0.0132  -0.0762  -0.0011   0.0758   0.1428   0.0008   0.2126   0.0211   0.1099
dimension |       11       12       13       14       15       16       17       18       19       20
coef_     |   0.0000   0.0015   0.0000  -0.0293   0.0014   0.0000   0.0365   0.0000   0.0064   0.0000
dimension |       21       22       23       24       25       26       27       28       29       30
coef_     |   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000  -0.0049   0.0673
dimension |       31       32       33       34       35       36       37       38       39       40
coef_     |   0.0203   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0003   0.0000
dimension |       41       42       43      

In [6]:
reg.summary(dim_search=[2])

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.1485   0.8515

Fitted Plug-in Estimations (Maximin Effects):

dimension |        1        2        3        4        5        6        7        8        9       10
coef_     |   0.0074  -0.0132  -0.0762  -0.0011   0.0758   0.1428   0.0008   0.2126   0.0211   0.1099
dimension |       11       12       13       14       15       16       17       18       19       20
coef_     |   0.0000   0.0015   0.0000  -0.0293   0.0014   0.0000   0.0365   0.0000   0.0064   0.0000
dimension |       21       22       23       24       25       26       27       28       29       30
coef_     |   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000  -0.0049   0.0673
dimension |       31       32       33       34       35       36       37       38       39       40
coef_     |   0.0203   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0003   0.0000
dimension |       41       42       43      

In [7]:
reg.predict()

{'pred': array([ 0.59980916,  0.0349804 , -0.13899486,  0.13319991, -0.68127574,
        -0.48184502, -0.17022069,  0.15677349, -0.77144535,  1.13837822,
         0.46048242,  0.57196422, -0.43519576, -0.55605458,  0.2963047 ,
        -0.16059679,  0.21675115, -0.36824958, -0.83239964, -0.49934856,
        -0.25562494,  0.42434761,  0.91744696,  0.23179753, -0.58859777,
         0.8151657 ,  0.06801001,  0.0849275 ,  0.83306642,  0.04003204,
        -0.03601859, -0.2058141 ,  0.34361413, -0.02040277, -0.85262232,
        -0.19408246, -0.2646345 , -0.55323176, -0.3640268 , -0.22893331,
         0.03681625, -0.67315086,  0.6576169 ,  1.21628339,  0.01113962,
         0.02714514, -0.71359756, -0.15495462,  0.47990769,  0.26907842,
         0.24443202,  0.09720009,  0.52310755,  0.05343414, -0.98189137,
        -1.00192819,  0.38985337,  1.31522366, -1.33654918,  0.35145247,
         0.24495861,  0.19222922, -0.09022656,  0.61649998,  0.37755096,
        -0.42229247, -0.76525802,  0.473443

## Low-dimensional Case

### Data Generating Process

In [8]:
# Set random seed for reproducibility
np.random.seed(0)

# number of groups
L = 2
# dimension
p = 5

# mean vector for source
mean_source = np.zeros(p)



# Covariance matrix for source
cov_source = np.diag(np.ones(p))

# 1st group's source data
n1 = 100
X1 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n1)
b1 = np.zeros(p)
b1[0:5] = np.arange(1, 6) / 20
Y1 = X1 @ b1 + np.random.normal(size=n1)

# 2nd group's source data
n2 = 100
X2 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n2)
b2 = np.arange(6,1,-1) / 20
Y2 = X2 @ b2 + np.random.normal(size=n2)

# Target data with covariate shift
n0 = 100
mean0 = np.zeros(p)
cov0 = np.copy(cov_source)


X0 = multivariate_normal.rvs(mean=mean0, cov=cov0, size=n0)
Xlist = [X1, X2]
Ylist = [Y1, Y2]


In [9]:
## Loading matrix
p = 5
n_loading = 20

# Initialize matrix
loading_mat = np.zeros((n_loading, p))

# Assign values as in R
loading_mat[0:5, 0] = 0.4 + np.random.normal(0, 0.1, size=5)  # Adding some noise
loading_mat[5:10, 1] = 0.8 + np.random.normal(0, 0.1, size=5)
loading_mat[10:15, 2] = 0.3 + np.random.normal(0, 0.1, size=5)
loading_mat[15:20, 3] = 0.7 + np.random.normal(0, 0.1, size=5)
loading_mat[0:5, 4] = 0.2 + np.random.normal(0, 0.1, size=5)

### Estimation ($\widehat{\beta^\mathbb{Q}}, \widehat{\omega \beta^\mathbb{Q}}, \widehat{\gamma}$)

In [10]:
reg = DRlm.Regression(f_learner = 'linear')
reg.fit(Xlist, Ylist, loading_mat, X0=X0)
reg.infer(M=500, alpha=0.05, alpha_thres=0.01)

In [11]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.2094   0.7906

Fitted Plug-in Estimations (Maximin Effects):

dimension |        1        2        3        4        5
coef_     |   0.2174   0.2334   0.2868   0.1783   0.0728

Fitted Debiased Estimations:

dimension |        1        2        3        4        5        6        7        8        9       10
coef_     |   0.0615   0.0806   0.1313   0.0800   0.0751   0.2343   0.1876   0.1732   0.1472   0.2037
dimension |       11       12       13       14       15       16       17       18       19       20
coef_     |   0.1061   0.0775   0.0544   0.0853   0.0621   0.1080   0.1230   0.1060   0.1292   0.1356

Confidence Intervals for each coefficient:

dimension |              1              2              3              4              5
CI        | (0.0105,0.1126) (0.0132,0.1479) (0.0172,0.2454) (0.0140,0.1460) (0.0129,0.1373)
dimension |              6              7              8              9            

In [12]:
reg.summary(dim_search=[1, 2, 3, 11, 18])

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.2094   0.7906

Fitted Plug-in Estimations (Maximin Effects):

dimension |        1        2        3        4        5
coef_     |   0.2174   0.2334   0.2868   0.1783   0.0728

Fitted Debiased Estimations:

dimension |        1        2        3       11       18
coef_     |   0.0615   0.0806   0.1313   0.1061   0.1060

Confidence Intervals for each coefficient:

dimension |              1              2              3             11             18
CI        | (0.0105,0.1126) (0.0132,0.1479) (0.0172,0.2454) (0.0299,0.1823) (-0.0055,0.2176)



### Prediction with $\widehat{\beta^\mathbb{Q}}$

In [13]:
reg.predict()

{'pred': array([-1.89133590e-01, -6.00494453e-02,  3.66588616e-01,  8.28261022e-02,
         3.63520965e-01,  6.65040643e-02,  2.95489012e-01, -2.56504173e-01,
         7.50157784e-01, -2.13639331e-02,  2.51969600e-01, -7.53502651e-01,
        -2.55593165e-02,  2.15295814e-01,  1.92598714e-01, -7.05395495e-01,
         1.39318486e-01, -1.84190367e-01, -1.21995126e-01, -1.23030601e-01,
        -4.46056677e-01, -4.76976396e-01, -2.23493970e-01, -3.34402980e-01,
        -4.40258314e-01,  4.56178301e-01, -2.63898180e-02, -1.96258410e-01,
        -2.18867902e-01, -7.75824021e-02,  1.07318839e+00, -6.53253106e-01,
         2.30804018e-01,  5.87040316e-04,  5.58963892e-01, -2.44833807e-01,
         7.86178825e-01,  1.35477524e-01,  1.75500876e-01, -6.60618549e-01,
        -4.67095712e-04,  7.26065480e-01, -6.51296542e-01, -1.00749117e+00,
        -1.30622530e-01,  2.19143552e-01, -8.10112516e-01,  1.85360800e-01,
         1.85612186e-01,  4.02907993e-02,  3.94387475e-01, -1.03326391e+00,
    

## Classification

## Data Setting

### Binary CLassification

In [14]:
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

n = 100; p = 5; L = 2; N = 1000
np.random.seed(123)
beta_list = [np.random.normal(0.5,0.25,p) for _ in range(L)]

Xlist = [np.random.normal(0, 1, (n, p)) for _ in range(L)]
X0 = np.random.normal(0.1, 1, (N, p))
logits_list = [
    X.dot(beta) - np.mean(X.dot(beta))
    for X, beta in zip(Xlist, beta_list)
    ]
probs_list = [sigmoid(logits) for logits in logits_list]
ylist = [(np.random.binomial(1, probs)) for probs in probs_list]



### Inference

In [20]:
cc = DRlm.Classification(f_learner='linear', w_learner='xgb')
cc.fit(Xlist,ylist,X0)
cc.infer()



In [21]:
cc.summary()

Model Summary:
Fitted Weights:

dimension |        1        2
weight_   |   0.7893   0.2107

Fitted Coefficients:

Class 2 coefficients:
dimension |        1        2        3        4        5
coef_     |   0.6742   0.5533   0.6736   0.2153   0.4303

Confidence Intervals for each coefficient:

Class 2 Confidence Intervals:
dimension |              1              2              3              4              5
CIs       | (0.015,1.931) (-0.141,1.557) (0.030,1.994) (-0.378,1.169) (-0.303,1.752)



In [ ]:
cc.summary()

Model Summary:
Fitted Weights:

dimension |        1        2
weight_   |   0.7582   0.2418

Fitted Coefficients:

Class 2 coefficients:
dimension |        1        2        3        4        5
coef_     |   0.6348   0.5051   0.5992   0.1976   0.4269

Confidence Intervals for each coefficient:

Class 2 Confidence Intervals:
dimension |              1              2              3              4              5
CIs       | (0.036,1.390) (-0.132,1.294) (-0.043,1.614) (-0.371,0.815) (-0.213,1.161)



In [ ]:
cc.summary(
    dim_search = [3,5]
)

Model Summary:
Fitted Weights:

dimension |        1        2
weight_   |   0.7582   0.2418

Fitted Coefficients:

Class 2 coefficients:
dimension |        3        5
coef_     |   0.5992   0.4269

Confidence Intervals for each coefficient:

Class 2 Confidence Intervals:
dimension |              3              5
CIs       | (-0.043,1.614) (-0.213,1.161)



### Predict

In [ ]:
cc.predict_proba() 

{'pred_proba': array([[0.3454345 , 0.6545655 ],
        [0.35878924, 0.64121076],
        [0.55292547, 0.44707453],
        ...,
        [0.13080139, 0.86919861],
        [0.68838669, 0.31161331],
        [0.21948744, 0.78051256]])}

In [ ]:
cc.predict()

{'pred': array([1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1,
        1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0,
        0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1,
        1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1,
        1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1,
        0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1,
        0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
        0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1,
        1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0,
        0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1,
        1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0,
        1, 0, 1, 1, 0, 1, 1, 1

### Multi-Classification $K=2$

In [22]:
def softmax(x):
    """
    Input: dim:n*(C-1)
    Output: softmax probabilities, dim:n*C
    
    """
    
    x_max = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

np.random.seed(123)
K = 2 # Number of classes
Xlist = [np.random.normal(0, 1, (n, p)) for _ in range(L)]
X0 = np.random.normal(0.1, 1, (N, p))
beta_list = [np.column_stack((np.zeros(p),np.random.normal(0, 0.25, (p,K)))) for _ in range(L)]
logits_list = [
    X.dot(beta) - np.mean(X.dot(beta))
    for X, beta in zip(Xlist, beta_list)
]
probs_list = [softmax(logits) for logits in logits_list]
ylist = [np.array([np.random.multinomial(1, probs[i, :]).tolist().index(1) for i in range(n)]) for probs in probs_list]


### Inference

In [33]:
cc = DRlm.Classification(f_learner='linear', w_learner='xgb')
cc.fit(Xlist,ylist,X0)
cc.infer()

In [34]:
cc.summary()

Model Summary:
Fitted Weights:

dimension |        1        2
weight_   |   0.6181   0.3819

Fitted Coefficients:

Class 2 coefficients:
dimension |        1        2        3        4        5
coef_     |   0.3285   0.1799   0.1016   0.1813   0.4365

Class 3 coefficients:
dimension |        1        2        3        4        5
coef_     |  -0.2821   0.5332   0.2476  -0.3323   0.2943

Confidence Intervals for each coefficient:

Class 2 Confidence Intervals:
dimension |              1              2              3              4              5
CIs       | (-0.937,1.663) (-1.221,1.463) (-1.106,1.363) (-0.954,1.484) (-0.540,1.808)

Class 3 Confidence Intervals:
dimension |              1              2              3              4              5
CIs       | (-1.738,0.907) (-0.632,1.912) (-0.909,1.601) (-1.408,0.787) (-0.684,1.867)



In [35]:
cc.summary(
    dim_search = [3,5], class_search=3
)

Model Summary:
Fitted Weights:

dimension |        1        2
weight_   |   0.6181   0.3819

Fitted Coefficients:

Class 3 coefficients:
dimension |        3        5
coef_     |   0.2476   0.2943

Confidence Intervals for each coefficient:

Class 3 Confidence Intervals:
dimension |              3              5
CIs       | (-0.909,1.601) (-0.684,1.867)



### Predict

In [ ]:
cc.predict_proba()

{'pred_proba': array([[0.18597781, 0.17658149, 0.63744071],
        [0.44065281, 0.12374689, 0.43560029],
        [0.25250153, 0.46645683, 0.28104164],
        ...,
        [0.18052818, 0.28350621, 0.53596561],
        [0.19853889, 0.60943075, 0.19203037],
        [0.12645667, 0.12713534, 0.74640799]])}

In [ ]:
cc.predict()

{'pred': array([2, 0, 1, 2, 2, 2, 2, 1, 0, 1, 1, 2, 1, 1, 0, 1, 0, 1, 2, 2, 2, 1,
        2, 2, 1, 2, 0, 2, 1, 0, 2, 2, 0, 2, 0, 1, 0, 1, 2, 0, 0, 2, 1, 1,
        2, 2, 1, 1, 1, 1, 2, 2, 1, 2, 1, 2, 0, 0, 1, 2, 2, 2, 1, 2, 2, 2,
        0, 1, 2, 2, 0, 1, 1, 2, 1, 1, 1, 0, 2, 1, 1, 0, 2, 1, 2, 2, 1, 0,
        2, 2, 1, 2, 2, 1, 1, 1, 2, 2, 0, 1, 2, 0, 0, 1, 2, 0, 2, 1, 0, 0,
        1, 1, 0, 0, 2, 2, 2, 1, 1, 2, 2, 1, 1, 1, 0, 0, 2, 2, 2, 2, 1, 2,
        0, 2, 2, 1, 1, 1, 0, 2, 0, 2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 0, 1, 2,
        2, 1, 1, 2, 1, 0, 1, 2, 2, 1, 0, 2, 2, 1, 0, 1, 0, 2, 1, 1, 0, 2,
        2, 2, 2, 1, 1, 0, 1, 2, 1, 0, 0, 2, 0, 0, 2, 2, 0, 1, 2, 2, 2, 1,
        1, 0, 0, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1, 0, 2, 1, 1, 2, 0, 2, 0, 1,
        1, 0, 2, 1, 2, 1, 2, 1, 2, 2, 2, 1, 2, 1, 0, 1, 2, 2, 1, 1, 1, 2,
        0, 2, 0, 2, 1, 1, 0, 2, 1, 1, 0, 1, 1, 2, 1, 1, 1, 2, 2, 2, 0, 0,
        0, 1, 1, 1, 1, 1, 0, 2, 0, 2, 1, 2, 2, 0, 2, 2, 2, 1, 0, 2, 1, 1,
        1, 0, 2, 2, 2, 2, 0, 1